# 04 — Agentes de IA

> **Bloque:** LLMs | **Nivel:** Intermedio-Avanzado
>
> Complementario al tutorial [04-agentes-ia.md](../../llms/04-agentes-ia.md)

Construimos un agente paso a paso: desde una herramienta simple hasta un bucle agéntico completo.

In [ ]:
# %pip install anthropic python-dotenv

import anthropic
import json
import os
import math
from datetime import datetime

client = anthropic.Anthropic()
MODELO = 'claude-sonnet-4-6'
print('✓ Listo')

## 1. Herramienta simple: calculadora

In [ ]:
tools_calculadora = [{
    'name': 'calcular',
    'description': 'Evalúa una expresión matemática y devuelve el resultado numérico',
    'input_schema': {
        'type': 'object',
        'properties': {
            'expresion': {'type': 'string', 'description': 'Expresión matemática (ej: 2**10, math.sqrt(144))'}
        },
        'required': ['expresion']
    }
}]

def calcular(expresion: str) -> dict:
    try:
        resultado = eval(expresion, {'__builtins__': {}}, {'math': math})
        return {'resultado': resultado, 'expresion': expresion}
    except Exception as e:
        return {'error': str(e)}

# Primera llamada: Claude decide usar la herramienta
pregunta = '¿Cuánto es la raíz cuadrada de 1764 elevada al cubo?'
print(f'Pregunta: {pregunta}\n')

r1 = client.messages.create(
    model=MODELO, max_tokens=512, tools=tools_calculadora,
    messages=[{'role': 'user', 'content': pregunta}]
)
print(f'Stop reason: {r1.stop_reason}')

if r1.stop_reason == 'tool_use':
    tool_use = next(b for b in r1.content if b.type == 'tool_use')
    print(f'Herramienta: {tool_use.name}({tool_use.input})')
    resultado = calcular(**tool_use.input)
    print(f'Resultado local: {resultado}')

    r2 = client.messages.create(
        model=MODELO, max_tokens=512, tools=tools_calculadora,
        messages=[
            {'role': 'user', 'content': pregunta},
            {'role': 'assistant', 'content': r1.content},
            {'role': 'user', 'content': [{
                'type': 'tool_result', 'tool_use_id': tool_use.id,
                'content': json.dumps(resultado)
            }]}
        ]
    )
    print(f'\nRespuesta: {r2.content[0].text}')

## 2. Múltiples herramientas

In [ ]:
TOOLS = [
    {
        'name': 'calcular',
        'description': 'Evalúa expresiones matemáticas',
        'input_schema': {'type': 'object', 'properties': {
            'expresion': {'type': 'string'}}, 'required': ['expresion']}
    },
    {
        'name': 'obtener_fecha',
        'description': 'Devuelve la fecha y hora actuales',
        'input_schema': {'type': 'object', 'properties': {}}
    },
    {
        'name': 'buscar_info',
        'description': 'Busca información en la base de conocimiento interna',
        'input_schema': {'type': 'object', 'properties': {
            'tema': {'type': 'string', 'description': 'Tema a buscar'}},
            'required': ['tema']}
    }
]

BASE_CONOCIMIENTO = {
    'anthropic': 'Anthropic es una empresa de seguridad en IA fundada en 2021.',
    'claude': 'Claude es la familia de modelos de IA de Anthropic.',
    'python': 'Python es el lenguaje dominante en IA y ciencia de datos.',
}

EJECUTORES = {
    'calcular': lambda expresion: calcular(expresion),
    'obtener_fecha': lambda: {'datetime': datetime.now().strftime('%Y-%m-%d %H:%M:%S')},
    'buscar_info': lambda tema: {
        'resultado': BASE_CONOCIMIENTO.get(tema.lower(), f'No se encontró información sobre: {tema}')
    },
}

print('✓ Herramientas definidas:', [t['name'] for t in TOOLS])

## 3. Bucle agéntico completo

In [ ]:
def agente(objetivo: str, max_pasos: int = 8) -> str:
    mensajes = [{'role': 'user', 'content': objetivo}]
    print(f'🎯 {objetivo}\n' + '─' * 50)

    for paso in range(1, max_pasos + 1):
        r = client.messages.create(
            model=MODELO, max_tokens=1024, tools=TOOLS,
            system='Eres un agente que resuelve tareas usando herramientas. '
                   'Razona paso a paso y usa las herramientas disponibles.',
            messages=mensajes
        )
        mensajes.append({'role': 'assistant', 'content': r.content})

        # Mostrar texto de razonamiento
        for b in r.content:
            if hasattr(b, 'text') and b.text:
                print(f'[Paso {paso}] 💭 {b.text[:200]}')

        if r.stop_reason == 'end_turn':
            texto = next((b.text for b in r.content if hasattr(b, 'text')), '')
            print(f'\n✅ Completado en {paso} paso(s)')
            return texto

        # Ejecutar herramientas
        resultados = []
        for b in r.content:
            if b.type != 'tool_use': continue
            print(f'[Paso {paso}] 🔧 {b.name}({b.input})')
            ejecutor = EJECUTORES.get(b.name)
            res = ejecutor(**b.input) if b.input else ejecutor()
            print(f'           📦 {res}')
            resultados.append({'type': 'tool_result', 'tool_use_id': b.id,
                                'content': json.dumps(res, ensure_ascii=False)})
        mensajes.append({'role': 'user', 'content': resultados})

    return '⚠️ Límite de pasos alcanzado'

# Prueba 1: cálculo simple
resultado = agente('¿Cuántos segundos hay en una semana?')

In [ ]:
# Prueba 2: múltiples herramientas en una tarea
resultado = agente('¿Qué fecha es hoy? Además, ¿cuántos días han pasado desde el 1 de enero de 2026 hasta hoy?')

In [ ]:
# Prueba 3: búsqueda + cálculo combinados
resultado = agente('Busca información sobre Anthropic y calcula cuántos años tiene la empresa a día de hoy.')

## 4. Tu propio agente

In [ ]:
# Añade tus propias herramientas al diccionario EJECUTORES y TOOLS
# y prueba con tu propio objetivo

mi_objetivo = 'Busca información sobre Python y dime cuántas letras tiene el nombre del lenguaje al cuadrado.'
resultado = agente(mi_objetivo)

---
**Tutorial relacionado:** [04-agentes-ia.md](../../llms/04-agentes-ia.md)